# Barcode Detection with OpenCV

This notebook tests OpenCV's built-in detectors as a decoder-led pipeline over the 300 DPI rendered pages. It mirrors the deterministic pipeline shape, but keeps OpenCV as the only decoder:

1. Full-page OpenCV decode.
2. Enhanced and scaled variants.
3. Rotated page decode for tilted labels.
4. Tiled decode for smaller regions.
5. OpenCV detect-only candidates, cropped and decoded again.
6. Merge duplicates, save JSON, and save review previews.

Research notes from the official OpenCV docs:

- `QRCodeDetector` supports single and multi-code calls such as `detectAndDecode`, `detectAndDecodeMulti`, `detect`, and `detectMulti`.
- `BarcodeDetector` lives under `cv2.barcode` and exposes `detectAndDecodeWithType` plus detect-only calls.
- OpenCV's barcode tutorial documents current 1D support as EAN-8, EAN-13, UPC-A, and UPC-E, so this is not a broad Code39/Code128/DataMatrix replacement.
- Sources: https://docs.opencv.org/4.x/de/dc3/classcv_1_1QRCodeDetector.html, https://docs.opencv.org/4.x/dc/df7/classcv_1_1barcode_1_1BarcodeDetector.html, https://docs.opencv.org/4.x/d6/d25/tutorial_barcode_detect_and_decode.html


In [ ]:
from pathlib import Path
import json
import subprocess
import shutil
import math

import cv2
import numpy as np
from PIL import Image, ImageDraw

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'pdfs').exists():
    ROOT = ROOT.parent

PDF_PATH = ROOT / 'data' / 'pdfs' / 'private-evaluation-document.pdf'
OUT_DIR = ROOT / 'output' / 'opencv-barcode-qrcode'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300

PAGES = list(range(1, 22))
EXPECTED_COUNTS = [0, 0, 1, 1, 1, 3, 1, 4, 1, 2, 1, 0, 0, 0, 0, 5, 2, 4, 6, 2, 9]

FULL_PAGE_SCALES = [1, 2]
ROTATED_PAGE_SCALES = [1]
TILE_SCALES = [1, 2]
PAGE_ROTATION_ANGLES = [5, 10, 15, -5, -10, -15]
TILE_SIZE = 1600
TILE_OVERLAP = 350
MAX_CANDIDATES_PER_PAGE = 80
INCLUDE_UNDECODED_REVIEW = False

GRID_COLS = 2
CELL_WIDTH = 650

QR_DETECTOR = cv2.QRCodeDetector()
BARCODE_DETECTOR = cv2.barcode.BarcodeDetector() if hasattr(cv2, 'barcode') else None

print('OpenCV:', cv2.__version__)
print('BarcodeDetector:', BARCODE_DETECTOR is not None)



Load/render pages and define geometry, enhancement, rotation, and merge helpers shared by all later stages.


In [ ]:
def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'barcode-research'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')


def enhance(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    equalized = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    sharpened = cv2.filter2D(equalized, -1, np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))
    return cv2.cvtColor(sharpened, cv2.COLOR_GRAY2BGR)


def rotate_bound(image, angle):
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos, sin = abs(matrix[0, 0]), abs(matrix[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    matrix[0, 2] += new_w / 2 - center[0]
    matrix[1, 2] += new_h / 2 - center[1]
    rotated = cv2.warpAffine(image, matrix, (new_w, new_h), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
    return rotated, cv2.invertAffineTransform(matrix)


def order_points(points):
    points = np.asarray(points, dtype='float32')
    s = points.sum(axis=1)
    diff = np.diff(points, axis=1).ravel()
    return np.array([
        points[np.argmin(s)],
        points[np.argmin(diff)],
        points[np.argmax(s)],
        points[np.argmax(diff)],
    ], dtype='float32')


def rect_from_box(box):
    xs = [point[0] for point in box]
    ys = [point[1] for point in box]
    return [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))]


def overlap_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    return inter / (area_a + area_b - inter + 1e-9)


def overlap_smaller(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    return inter / (min(area_a, area_b) + 1e-9)


def rect_center(rect):
    x1, y1, x2, y2 = rect
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def center_distance(a, b):
    ax, ay = rect_center(a)
    bx, by = rect_center(b)
    return float(np.hypot(ax - bx, ay - by))


def scaled_variants(image, scales):
    for scale in scales:
        scaled = image if scale == 1 else cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        yield f'orig@{scale}x', scaled, scale
        yield f'enhanced@{scale}x', enhance(scaled), scale


def as_boxes(points, scale=1.0, offset=(0, 0)):
    if points is None:
        return []
    points = np.asarray(points, dtype=np.float32)
    if points.size == 0:
        return []
    boxes = points.reshape(-1, 4, 2) / scale
    ox, oy = offset
    boxes[:, :, 0] += ox
    boxes[:, :, 1] += oy
    return boxes.astype(float)


def add_unique(detections, candidate):
    for existing in detections:
        same_text = candidate.get('raw_value') and candidate.get('raw_value') == existing.get('raw_value')
        if overlap_iou(candidate['rect'], existing['rect']) > 0.35:
            return
        if same_text and overlap_smaller(candidate['rect'], existing['rect']) > 0.15:
            return
        if same_text:
            c_rect, e_rect = candidate['rect'], existing['rect']
            c_long = max(c_rect[2] - c_rect[0], c_rect[3] - c_rect[1])
            e_long = max(e_rect[2] - e_rect[0], e_rect[3] - e_rect[1])
            if center_distance(c_rect, e_rect) < max(100.0, min(c_long, e_long) * 0.25):
                return
    detections.append(candidate)



Wrap the OpenCV QR and barcode APIs so all pipeline layers return the same detection shape.


In [ ]:
def opencv_decode(image, scale=1.0, offset=(0, 0)):
    detections = []

    found, decoded, points, _ = QR_DETECTOR.detectAndDecodeMulti(image)
    if found:
        for text, box in zip(decoded, as_boxes(points, scale=scale, offset=offset)):
            if text:
                detections.append({'format': 'QRCode', 'raw_value': text, 'box': box.tolist()})
    else:
        text, points, _ = QR_DETECTOR.detectAndDecode(image)
        for box in as_boxes(points, scale=scale, offset=offset):
            if text:
                detections.append({'format': 'QRCode', 'raw_value': text, 'box': box.tolist()})

    if BARCODE_DETECTOR is not None:
        found, decoded, types, points = BARCODE_DETECTOR.detectAndDecodeWithType(image)
        if found:
            for text, fmt, box in zip(decoded, types, as_boxes(points, scale=scale, offset=offset)):
                if text:
                    detections.append({'format': fmt or 'barcode', 'raw_value': text, 'box': box.tolist()})

    return detections


def decoded_detections(image, source, offset=(0, 0), box_override=None, scales=None):
    detections = []
    for variant_name, variant, scale in scaled_variants(image, scales or [1]):
        for det in opencv_decode(variant, scale=scale, offset=offset):
            box = box_override if box_override is not None else det['box']
            detections.append({
                'source': source,
                'variant': variant_name,
                'format': det['format'],
                'raw_value': det['raw_value'],
                'decoded': True,
                'score': 100.0 if source.startswith('full') else 80.0 if source.startswith('tile') else 70.0,
                'box': [[float(x), float(y)] for x, y in box],
                'rect': rect_from_box(box),
            })
    return detections


def full_page_decode(image):
    detections = []
    for det in decoded_detections(image, 'full-page', scales=FULL_PAGE_SCALES):
        add_unique(detections, det)
    return detections


def tile_decode(image):
    detections = []
    h, w = image.shape[:2]
    step = max(1, TILE_SIZE - TILE_OVERLAP)
    for y in range(0, h, step):
        for x in range(0, w, step):
            tile = image[y:min(y + TILE_SIZE, h), x:min(x + TILE_SIZE, w)]
            if tile.shape[0] < 250 or tile.shape[1] < 250:
                continue
            for det in decoded_detections(tile, 'tile', offset=(x, y), scales=TILE_SCALES):
                add_unique(detections, det)
    return detections


def rotated_page_decode(image):
    detections = []
    for angle in PAGE_ROTATION_ANGLES:
        rotated, inverse = rotate_bound(image, angle)
        for variant_name, variant, scale in scaled_variants(rotated, ROTATED_PAGE_SCALES):
            for det in opencv_decode(variant, scale=scale):
                box = [[float((inverse @ [x, y, 1])[0]), float((inverse @ [x, y, 1])[1])] for x, y in det['box']]
                candidate = {
                    'source': 'rotated-page',
                    'variant': variant_name,
                    'rotation_angle': angle,
                    'format': det['format'],
                    'raw_value': det['raw_value'],
                    'decoded': True,
                    'score': 70.0,
                    'box': box,
                    'rect': rect_from_box(box),
                }
                add_unique(detections, candidate)
    return detections


Use OpenCV detect-only calls as candidates, then crop/deskew those boxes and decode the crops again.


In [ ]:
def detect_boxes(image):
    boxes = []
    for kind, detector in [('QR-candidate', QR_DETECTOR), ('barcode-candidate', BARCODE_DETECTOR)]:
        if detector is None:
            continue
        ok, points = detector.detectMulti(image)
        if not ok:
            ok, points = detector.detect(image)
        for box in as_boxes(points):
            boxes.append({
                'source': kind,
                'format': 'candidate',
                'raw_value': '',
                'decoded': False,
                'score': 20.0,
                'box': box.tolist(),
                'rect': rect_from_box(box),
            })
    return boxes


def crop_quad(image, points, pad=0.12):
    points = order_points(points)
    cx, cy = points[:, 0].mean(), points[:, 1].mean()
    points = np.column_stack([
        cx + (points[:, 0] - cx) * (1 + pad),
        cy + (points[:, 1] - cy) * (1 + pad),
    ]).astype('float32')

    tl, tr, br, bl = points
    width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    if width < 2 or height < 2:
        return None

    dest = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype='float32')
    matrix = cv2.getPerspectiveTransform(points, dest)
    crop = cv2.warpPerspective(image, matrix, (width, height), borderValue=(255, 255, 255))
    if crop.shape[0] > crop.shape[1]:
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
    return crop


def opencv_candidates(image):
    candidates = []
    for det in detect_boxes(image):
        if det['rect'][2] - det['rect'][0] < 20 or det['rect'][3] - det['rect'][1] < 20:
            continue
        add_unique(candidates, det)
    return candidates[:MAX_CANDIDATES_PER_PAGE]


def decode_candidate_crops(image, existing):
    decoded = []
    for candidate in opencv_candidates(image):
        if any(overlap_iou(candidate['rect'], det['rect']) > 0.25 for det in existing + decoded):
            continue
        crop = crop_quad(image, candidate['box'])
        if crop is None:
            continue
        for det in decoded_detections(crop, 'candidate-crop', box_override=candidate['box'], scales=[1, 2]):
            det['score'] = 60.0
            add_unique(decoded, det)
    return decoded


Combine layers, draw review previews, and save the same kind of artifacts as the deterministic pipeline.


In [ ]:
def detect_page(image):
    detections = []
    for layer in (full_page_decode, rotated_page_decode, tile_decode):
        for det in layer(image):
            add_unique(detections, det)
    for det in decode_candidate_crops(image, detections):
        add_unique(detections, det)
    if INCLUDE_UNDECODED_REVIEW:
        for det in opencv_candidates(image):
            add_unique(detections, det)
    return sorted(detections, key=lambda d: d['score'], reverse=True)


def draw_detections(image_path, detections):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    colors = ['red', 'lime', 'blue', 'orange', 'magenta', 'cyan', 'yellow', 'purple', 'pink']

    for i, det in enumerate(detections, start=1):
        points = [tuple(point) for point in det['box']]
        color = colors[(i - 1) % len(colors)]
        draw.line(points + [points[0]], fill=color, width=7)
        x = min(p[0] for p in points)
        y = max(0, min(p[1] for p in points) - 28)
        value = det.get('raw_value') or 'undecoded'
        label = f'{i} {det["source"]} {det.get("format", "?")} {value[:22]}'
        text_w = max(260, min(560, 9 * len(label)))
        draw.rectangle([x, y, x + text_w, y + 28], fill=color)
        draw.text((x + 4, y + 6), label, fill='black')
    return image


def create_grid(images, cols=2, cell_width=650):
    if not images:
        return None
    thumbs = []
    for image in images:
        img = image.copy()
        ratio = cell_width / img.width
        thumbs.append(img.resize((cell_width, int(img.height * ratio))))
    rows = math.ceil(len(thumbs) / cols)
    cell_height = max(img.height for img in thumbs)
    grid = Image.new('RGB', (cols * cell_width, rows * cell_height), 'white')
    for i, img in enumerate(thumbs):
        grid.paste(img, ((i % cols) * cell_width, (i // cols) * cell_height))
    return grid


def json_ready(detections):
    clean = []
    for det in detections:
        item = dict(det)
        item['box'] = [[float(x), float(y)] for x, y in item['box']]
        item['rect'] = [float(v) for v in item['rect']]
        clean.append(item)
    return clean


Run the OpenCV pipeline on all 21 pages and write the review artifacts.


In [ ]:
render_pdf_if_needed()
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'pages').mkdir(exist_ok=True)

results = []
preview_images = []

for page in PAGES:
    image_path = PAGE_DIR / f'page-{page:02d}.png'
    image_cv = cv2.imread(str(image_path))
    expected = EXPECTED_COUNTS[page - 1]
    detections = [] if image_cv is None else detect_page(image_cv)

    debug = draw_detections(image_path, detections)
    debug_path = OUT_DIR / 'pages' / f'page-{page:02d}.png'
    debug.save(debug_path)
    preview_images.append(debug)

    decoded_count = sum(1 for det in detections if det.get('decoded'))
    print(f'page {page:02d}: decoded={decoded_count:2d} total={len(detections):2d} expected={expected:2d}')
    for det in detections:
        if det.get('decoded'):
            print(f'  [{det["format"]}] {det["raw_value"]} ({det["source"]})')

    results.append({
        'page': page,
        'expected': expected,
        'decoded_count': decoded_count,
        'total_detections': len(detections),
        'detections': json_ready(detections),
        'debug_image': str(debug_path),
    })

(OUT_DIR / 'detections.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
grid = create_grid(preview_images, cols=GRID_COLS, cell_width=CELL_WIDTH)
grid_path = OUT_DIR / 'all-pages-grid.png'
grid.save(grid_path)
print(f'\nSaved JSON: {OUT_DIR / "detections.json"}')
print(f'Saved grid: {grid_path}')
display(grid)


## Inspect One Page

Open one generated page preview at full size for quick inspection.


In [ ]:
PAGE_TO_VIEW = 6
display(Image.open(OUT_DIR / 'pages' / f'page-{PAGE_TO_VIEW:02d}.png'))
